# Phase 6 — Per-Segment Timing Statistics (All 7 Methods)

Revision Round 2 — addresses R1 request for timing variance.

Runs all 7 detection methods on every segment of the DroneRF dataset and records
per-segment wall-clock time. Computes mean ± std and 95% CI for each method.

**Methods timed:**
- Energy, Wavelet, Cyclo — standalone, CPU (NumPy)
- Cascade — energy stage + burst aggregation + selective SCF, CPU
- AED-global — energy + background z-score, CPU
- AED-perseg — energy + within-segment MAD normalization, CPU
- CNN — PSD feature extraction (CPU, NumPy FFT) + LightweightCNN inference (GPU, batch=256)

**CNN timing note:** A freshly instantiated (randomly weighted) LightweightCNN is used.
Model weights do not affect inference time — only architecture and batch size matter.
`torch.cuda.synchronize()` is called before stopping the GPU timer.

**Output:** `results/review_1/timing_stats.csv`

**Expected runtime:** ~12 min (bottleneck: Cyclo at ~2 s/segment × 227 segments)

In [1]:
import os
import sys
import time
import numpy as np
import pandas as pd
from glob import glob
from pathlib import Path
from dotenv import load_dotenv
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

ROOT = Path().resolve().parents[2]
sys.path.insert(0, str(ROOT / "src"))

from rfml_uav.drone_rf.consts import BUI, FS_HZ
from rfml_uav.drone_rf.utils import get_segment_count, get_binary_label
from rfml_uav.signal_detection.methods import (
    energy_detector, wavelet_detector, cyclo_detector,
)
from rfml_uav.signal_detection.cascade import cascade_detector

load_dotenv()
CHUNKED_PATH = Path(os.getenv("DATA_PATH")) / "chunked"
OUT = ROOT / "results" / "review_1"
OUT.mkdir(parents=True, exist_ok=True)

DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_CHUNK       = 4000
N_FREQ        = N_CHUNK // 2 + 1  # 2001
BATCH_SIZE    = 256
MIN_BURST_LEN = 50
FIXED_PFA     = 0.05

SIGNAL_BUIS = [b for b in BUI if get_binary_label(b) == 1]
NOISE_BUIS  = [b for b in BUI if get_binary_label(b) == 0]
ALL_BUIS    = [(b, 1) for b in SIGNAL_BUIS] + [(b, 0) for b in NOISE_BUIS]

print(f"ROOT         : {ROOT}")
print(f"CHUNKED_PATH : {CHUNKED_PATH}")
print(f"Device       : {DEVICE}")

ROOT         : /home/ivan/work/ml/rfml-uav
CHUNKED_PATH : /home/ivan/work/ml/rfml-uav/data/drone-rf/chunked
Device       : cuda


---
## CNN model + PSD helper

Fresh randomly-initialized LightweightCNN — weights don't affect timing.

In [2]:
class LightweightCNN(nn.Module):
    def __init__(self, n_freq=N_FREQ):
        super().__init__()
        self.conv1 = nn.Conv1d(1,  16, kernel_size=11, padding=5)
        self.conv2 = nn.Conv1d(16, 32, kernel_size=7,  padding=3)
        self.gap   = nn.AdaptiveAvgPool1d(1)
        self.fc1   = nn.Linear(32, 64)
        self.drop  = nn.Dropout(0.3)
        self.fc2   = nn.Linear(64, 1)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.gap(x).squeeze(-1)
        x = F.relu(self.fc1(x))
        x = self.drop(x)
        return self.fc2(x).squeeze(-1)


def compute_psd(chunks):
    """Batch log-magnitude PSD on CPU (np.fft.rfft). Returns float32 (N, N_FREQ)."""
    return np.log1p(np.abs(np.fft.rfft(chunks, axis=1))).astype(np.float32)


cnn_model = LightweightCNN().to(DEVICE)
cnn_model.eval()

# GPU warm-up — trigger JIT/cuDNN init before timing starts
with torch.no_grad():
    _wu = torch.zeros(BATCH_SIZE, 1, N_FREQ, device=DEVICE)
    _ = cnn_model(_wu)
if DEVICE.type == "cuda":
    torch.cuda.synchronize()
print(f"LightweightCNN ready on {DEVICE} ({sum(p.numel() for p in cnn_model.parameters()):,} params)")

LightweightCNN ready on cuda (5,985 params)


---
## Background calibration

Energy threshold and AED background statistics from BUI `00000` (background only).

In [3]:
print("Calibrating background...")
bg_scores = []
for seg in range(get_segment_count("00000")):
    for fp in sorted(glob(str(CHUNKED_PATH / f"00000*_{seg}.npz"))):
        chunks = np.load(fp)["arr_0"]
        for x in chunks:
            bg_scores.append(energy_detector(x))

bg_arr     = np.array(bg_scores)
thr_energy = np.percentile(bg_arr, 100 * (1 - FIXED_PFA))
AED_MU     = np.median(bg_arr)
AED_SIGMA  = max(np.median(np.abs(bg_arr - AED_MU)) * 1.4826, 1e-10)

print(f"Background chunks : {len(bg_arr):,}")
print(f"Energy threshold  : {thr_energy:.4f}")
print(f"AED mu / sigma    : {AED_MU:.4f} / {AED_SIGMA:.6f}")

Calibrating background...
Background chunks : 205,000
Energy threshold  : 14.7695
AED mu / sigma    : 5.9463 / 0.278358


---
## Timing loop — all 7 methods, all segments

Each method is timed independently on the same loaded data.
Per-segment time = sum over all .npz files in the segment.

CNN timing is split into PSD (CPU) and inference (GPU) for reporting,
but the total is used in the final table.

In [4]:
METHODS = ["Energy", "Wavelet", "Cyclo", "Cascade",
           "AED-global", "AED-perseg", "CNN_psd", "CNN_inf", "CNN"]
timings = {m: [] for m in METHODS}

for bui, label in tqdm(ALL_BUIS, desc="Timing"):
    for seg in range(get_segment_count(bui)):
        files = sorted(glob(str(CHUNKED_PATH / f"{bui}*_{seg}.npz")))
        if not files:
            continue

        t_energy = t_wavelet = t_cyclo = t_cascade = 0.0
        t_aed_g  = t_aed_p  = t_psd   = t_cnn_inf = 0.0

        for fp in files:
            data = np.load(fp)["arr_0"]  # (N, N_CHUNK)

            # Energy
            t0 = time.perf_counter()
            e_scores = [energy_detector(x) for x in data]
            t_energy += time.perf_counter() - t0

            # Wavelet
            t0 = time.perf_counter()
            _ = [wavelet_detector(x) for x in data]
            t_wavelet += time.perf_counter() - t0

            # Cyclo
            t0 = time.perf_counter()
            _ = [cyclo_detector(x, fs=FS_HZ) for x in data]
            t_cyclo += time.perf_counter() - t0

            # Cascade (energy stage + aggregation + selective SCF)
            t0 = time.perf_counter()
            cascade_detector(
                chunks=data, fs=FS_HZ,
                thr=thr_energy, min_burst_len=MIN_BURST_LEN)
            t_cascade += time.perf_counter() - t0

            # AED-global (energy + linear rescale)
            t0 = time.perf_counter()
            e_arr = np.array([energy_detector(x) for x in data])
            _     = (e_arr - AED_MU) / AED_SIGMA
            t_aed_g += time.perf_counter() - t0

            # AED-perseg (energy + within-segment MAD normalization)
            t0    = time.perf_counter()
            e_arr = np.array([energy_detector(x) for x in data])
            mu_p  = np.percentile(e_arr, 25)
            mad_p = np.median(np.abs(e_arr - np.median(e_arr)))
            sig_p = max(mad_p * 1.4826, 1e-10)
            _     = (e_arr - mu_p) / sig_p
            t_aed_p += time.perf_counter() - t0

            # CNN — PSD (CPU)
            t0  = time.perf_counter()
            psd = compute_psd(data)
            t_psd += time.perf_counter() - t0

            # CNN — inference (GPU)
            t0 = time.perf_counter()
            with torch.no_grad():
                for i in range(0, len(psd), BATCH_SIZE):
                    batch  = psd[i:i + BATCH_SIZE]
                    x_gpu  = torch.from_numpy(batch).unsqueeze(1).to(DEVICE)
                    _      = cnn_model(x_gpu)
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            t_cnn_inf += time.perf_counter() - t0

        timings["Energy"].append(t_energy)
        timings["Wavelet"].append(t_wavelet)
        timings["Cyclo"].append(t_cyclo)
        timings["Cascade"].append(t_cascade)
        timings["AED-global"].append(t_aed_g)
        timings["AED-perseg"].append(t_aed_p)
        timings["CNN_psd"].append(t_psd)
        timings["CNN_inf"].append(t_cnn_inf)
        timings["CNN"].append(t_psd + t_cnn_inf)

n_segs = len(timings["Energy"])
print(f"\nDone. Timed {n_segs} segments.")

Timing: 100%|██████████| 10/10 [12:49<00:00, 76.93s/it]


Done. Timed 227 segments.


---
## Statistics and output

In [5]:
REPORT_METHODS = [
    "Energy", "Wavelet", "Cyclo", "Cascade",
    "AED-global", "AED-perseg", "CNN",
]

rows = []
print(f"{'Method':<14} {'Mean (ms)':>10} {'Std (ms)':>10} {'95% CI (ms)':>13} {'n':>5}")
print("-" * 57)
for method in REPORT_METHODS:
    arr  = np.array(timings[method]) * 1e3
    mean = arr.mean()
    std  = arr.std(ddof=1)
    ci95 = 1.96 * std / np.sqrt(len(arr))
    n    = len(arr)
    print(f"{method:<14} {mean:>10.2f} {std:>10.2f} {ci95:>13.2f} {n:>5}")
    rows.append({"Method": method, "Mean_ms": round(mean, 2),
                 "Std_ms": round(std, 2), "CI95_ms": round(ci95, 2), "N": n})

df = pd.DataFrame(rows)
df.to_csv(OUT / "timing_stats.csv", index=False)
print(f"\nSaved: timing_stats.csv")

# Also print CNN PSD vs inference breakdown
print("\nCNN breakdown (mean ms):")
print(f"  PSD computation (CPU) : {np.mean(timings['CNN_psd'])*1e3:.2f} ms")
print(f"  Inference ({DEVICE})  : {np.mean(timings['CNN_inf'])*1e3:.2f} ms")

Method          Mean (ms)   Std (ms)   95% CI (ms)     n
---------------------------------------------------------
Energy              19.78      15.45          2.01   227
Wavelet            173.96     134.72         17.53   227
Cyclo             2094.86    1619.12        210.63   227
Cascade            849.96    1022.95        133.08   227
AED-global          20.32      15.49          2.01   227
AED-perseg          19.84      15.08          1.96   227
CNN                 78.01      63.15          8.22   227

Saved: timing_stats.csv

CNN breakdown (mean ms):
  PSD computation (CPU) : 55.32 ms
  Inference (cuda)  : 22.69 ms
